# Data Smoothing in Feature Engineering

## 1. Clear Overview

Smoothing is a data preprocessing technique designed to reduce random noise and minor fluctuations in a dataset, thereby highlighting the true underlying trend or signal. By applying smoothing, models can learn more robust patterns and improve their ability to generalize, rather than over-fitting to random variations.

In [ ]:
# Always start with imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set display options for pandas
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("Step 1: All necessary libraries successfully imported!")

## 2. Structured Table of Contents

- **Synthetic Data Creation**: Building our noisy time-series dataset
- **Purpose of Smoothing**: Why we use it and its benefits
- **Core Concept 1**: Simple Moving Average (SMA)
- **Core Concept 2**: Weighted Moving Average (WMA)
- **Core Concept 3**: Exponential Smoothing (EMA)
- **Core Concept 4**: Expanding Window Average
- **Evaluating Model Impact**: Measuring signal recovery
- **Handling Missing Values**: Dealing with window gaps
- **Practice Exercises**: Apply what you've learned
- **Summary**: Key takeaways

## 3. Synthetic Data Creation

To properly demonstrate smoothing, we need a dataset with a clear underlying signal but significant noise. We will generate 365 days of synthetic daily sales data. 

Our data will structurally contain:
1. A steady upward linear trend
2. A monthly seasonality pattern (modeled via a sine wave)
3. Random daily volatility (noise)

In [ ]:
# Step 2: Create synthetic daily sales data
days = np.arange(1, 366)
dates = pd.date_range(start="2023-01-01", periods=365, freq="D")

# 1. Base trend: Steady growth over the year
trend = days * 0.5

# 2. Seasonality: Monthly cycles (30 days)
seasonality = 50 * np.sin(2 * np.pi * days / 30)

# 3. Noise: Random spikes and drops representing daily volatility
noise = np.random.normal(loc=0, scale=30, size=365)

# Combine to form the true signal and the noisy observation
true_signal = 200 + trend + seasonality
observed_sales = true_signal + noise

# Assemble DataFrame
df = pd.DataFrame({
    'date': dates,
    'day_index': days,
    'true_signal': true_signal,
    'raw_sales': observed_sales
})

print("Synthetic Sales Dataset Created!\n")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())

## 4. Why Use Smoothing?

- **Noise Reduction:** It irons out wrinkles (random spikes or drops) that can confuse machine learning algorithms.
- **Improved Generalization:** By focusing on the broader trend, models become less prone to memorizing noise, preventing overfitting.
- **Pattern Clarity:** It makes trends in complex or messy datasets much easier to visualize and explain to stakeholders.
- **Predictive Performance:** In time-series data, smoothing helps autoregressive models understand the genuine directional momentum.

Let's visualize the chaos of our raw data compared to the true underlying signal we want our model to learn.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df['date'], df['raw_sales'], color='skyblue', alpha=0.8, label='Raw Noisy Sales')
plt.plot(df['date'], df['true_signal'], color='black', linewidth=2, linestyle='--', label='True Underlying Signal')

plt.title('Daily Sales: Raw Data vs True Signal', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Sales Volume', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Diagnostic: The raw data fluctuates wildly around the true signal. If fed directly to a model, the algorithm might over-index on the noise.")

## 5. Core Concept 1: Simple Moving Average (SMA)

The **Simple Moving Average** replaces a data point with the unweighted mean of its neighbors within a predefined window. For example, a 7-day moving average smooths out weekly volatility by averaging values from the current day and the preceding 6 days.

Formula (7-day):
SMA = (Day1 + Day2 + Day3 + Day4 + Day5 + Day6 + Day7) / 7

- **Strength:** Extremely simple to compute and perfectly intuitive.
- **Weakness:** Gives equal weight to older data and recent data, which introduces a 'lag' in detecting sudden, genuine shifts.

In [ ]:
# Apply 7-Day Simple Moving Average using pandas rolling()
df['sma_7'] = df['raw_sales'].rolling(window=7).mean()

print("Calculated 7-Day Simple Moving Average.")
print("Notice that the first 6 rows are NaN because a full 7-day look-back window isn't available yet.")
print(df[['date', 'raw_sales', 'sma_7']].head(10))

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df['date'], df['raw_sales'], color='lightgray', label='Raw Sales')
plt.plot(df['date'], df['sma_7'], color='orange', linewidth=2.5, label='7-Day SMA')

plt.title('Smoothing Effect of 7-Day Simple Moving Average', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Sales Volume', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Insight: The orange line is much smoother and begins to clearly reveal the monthly sine-wave seasonality.")

## 6. Core Concept 2: Weighted Moving Average (WMA)

The **Weighted Moving Average** assigns different mathematical weights to values within the window. Typically, more recent values are given higher weight, while older values are given less weight. This helps mitigate the lag effect seen in simple moving averages.

Formula (3-day example):
WMA = (Day3 * 3 + Day2 * 2 + Day1 * 1) / (3 + 2 + 1)

Let's implement a 7-day WMA where today has a weight of 7, yesterday 6, all the way down to 1.

In [ ]:
# Define linearly increasing weights [1, 2, 3, 4, 5, 6, 7]
weights = np.arange(1, 8)

def calculate_wma(window_data):
    # Dot product of data and weights, divided by sum of weights
    return np.dot(window_data, weights) / weights.sum()

# Apply using pandas rolling().apply()
df['wma_7'] = df['raw_sales'].rolling(window=7).apply(calculate_wma, raw=True)

print("Calculated 7-Day Weighted Moving Average.")
print(df[['date', 'raw_sales', 'sma_7', 'wma_7']].iloc[5:10])

## 7. Core Concept 3: Exponential Smoothing (EMA)

**Exponential Smoothing (Exponential Moving Average)** is an advanced method where the influence of older observations decreases exponentially over time. This gives the absolute highest importance to the most recent data points and gracefully 'forgets' the past without a strict, rigid cut-off window.

EMA responds much faster to sudden anomalies compared to SMA.

Formula:
EMA_today = (Value_today * alpha) + (EMA_yesterday * (1 - alpha))

In [ ]:
# Apply Exponential Moving Average using pandas ewm()
# span=7 roughly corresponds to a 7-day moving window in terms of weight distribution
df['ema_7'] = df['raw_sales'].ewm(span=7, adjust=False).mean()

print("Calculated 7-Day Exponential Moving Average.")
print("Notice that EMA doesn't produce NaNs at the beginning; it starts immediately with the first value.")
print(df[['date', 'raw_sales', 'sma_7', 'wma_7', 'ema_7']].head(10))

## 8. Core Concept 4: Expanding Window Average

Instead of a rolling window that moves forward and drops old data, an **Expanding Window** includes ALL data from the beginning of the dataset up to the current point. This is effectively a cumulative average.

This is useful for tracking lifetime performance metrics (e.g., historical win-rate of a sports team up to today).

In [ ]:
# Apply expanding average using pandas expanding()
df['expanding_avg'] = df['raw_sales'].expanding().mean()

plt.figure(figsize=(14, 5))
plt.plot(df['date'], df['raw_sales'], color='lightgray', label='Raw Sales')
plt.plot(df['date'], df['expanding_avg'], color='purple', linewidth=3, label='Expanding (Cumulative) Average')

plt.title('Expanding Window Average', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Sales Volume')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Insight: The expanding average stabilizes over time, showing the absolute macroscopic center of the dataset.")

## 9. Visualization Gallery: Comparing the Techniques

Let's zoom in on a 60-day window to clearly see the distinct behavioral differences between SMA, WMA, and EMA. Watch how closely each line tracks the recent peaks and valleys.

In [ ]:
# Zoom in on a 60-day slice to see the differences clearly
df_subset = df.iloc[30:90]

plt.figure(figsize=(15, 7))
plt.plot(df_subset['date'], df_subset['raw_sales'], color='lightgray', marker='o', alpha=0.5, label='Raw Data')
plt.plot(df_subset['date'], df_subset['sma_7'], color='blue', linewidth=2, label='SMA (7-Day)')
plt.plot(df_subset['date'], df_subset['wma_7'], color='green', linewidth=2, label='WMA (7-Day)')
plt.plot(df_subset['date'], df_subset['ema_7'], color='red', linewidth=2, linestyle='--', label='EMA (Span=7)')

plt.title('Smoothing Methods Comparison (60-Day Zoom)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Sales Volume', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Observation: EMA (red) and WMA (green) hug the recent changes tighter than the SMA (blue).")
print("SMA suffers the most from 'lag' during sudden shifts in momentum.")

## 10. Evaluating Model Impact

How well did these smoothing methods recover the hidden True Signal we mathematically generated at the beginning? 
We can measure this objectively by calculating the Mean Absolute Error (MAE) between our smoothed features and the true underlying signal.

In [ ]:
# Drop initial NaNs for a perfectly fair, aligned comparison
eval_df = df.dropna().copy()

# Calculate Mean Absolute Error compared to the TRUE hidden signal
mae_raw = np.abs(eval_df['raw_sales'] - eval_df['true_signal']).mean()
mae_sma = np.abs(eval_df['sma_7'] - eval_df['true_signal']).mean()
mae_wma = np.abs(eval_df['wma_7'] - eval_df['true_signal']).mean()
mae_ema = np.abs(eval_df['ema_7'] - eval_df['true_signal']).mean()

print("--- Mean Absolute Error (vs True Underlying Signal) ---")
print(f"Raw Data Error: {mae_raw:.2f}")
print(f"SMA-7 Error:    {mae_sma:.2f}")
print(f"WMA-7 Error:    {mae_wma:.2f}")
print(f"EMA-7 Error:    {mae_ema:.2f}")

print("\nTakeaway: Smoothing significantly reduced the error. The SMA actually performed best at recovering the broad, slow-moving sine wave because it filtered out the most noise entirely, even if it lagged slightly.")

## 11. Handling Missing Values

Moving averages naturally produce missing values (`NaN`) at the beginning of the dataset because the initial rows lack historical context to form a complete window. Machine learning models cannot accept NaNs, so we must safely handle them.

In [ ]:
# Display the NaNs
print("Missing values before handling:")
print(df[['sma_7', 'wma_7']].head(8))

# Technique 1: Backfilling (carrying the first valid observation backwards)
df['sma_7_filled'] = df['sma_7'].fillna(method='bfill')

# Technique 2: Dropping (If dataset is large enough, simply drop the first N rows)
df_clean = df.dropna(subset=['wma_7'])

print("\nMissing values in SMA after Backfill:", df['sma_7_filled'].isna().sum())
print("Rows remaining after DropNA:", len(df_clean))

## 12. Practice Exercises

Now it is your turn to apply smoothing techniques. The size of the window drastically changes the feature representation.

### Exercise 1: Macro-Trend Extraction

**Task:** 
Create a **30-day Simple Moving Average** column named `sma_30`. A 30-day window should theoretically smooth out the entire 30-day seasonality (the sine wave), leaving behind only the pure linear growth trend.

In [ ]:
# --- EXERCISE 1 SOLUTION ---

df['sma_30'] = df['raw_sales'].rolling(window=30).mean()

print("30-Day SMA Calculated!")
print(f"Number of initial NaNs generated: {df['sma_30'].isna().sum()}")

### Exercise 2: Visualizing the Macro Trend

**Task:** 
Plot the `raw_sales`, the 7-day `sma_7`, and your new 30-day `sma_30` on a single chart to see how radically different window sizes alter the feature.

In [ ]:
# --- EXERCISE 2 SOLUTION ---

plt.figure(figsize=(15, 6))
plt.plot(df['date'], df['raw_sales'], color='lightgray', alpha=0.5, label='Raw Data')
plt.plot(df['date'], df['sma_7'], color='orange', linewidth=2, label='7-Day SMA (Micro Trend)')
plt.plot(df['date'], df['sma_30'], color='darkblue', linewidth=3, label='30-Day SMA (Macro Trend)')

plt.title('Impact of Window Size on Smoothing Features', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Sales Volume')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Insight: A 30-day window completely erases the 30-day seasonal sine wave, exposing the straight upward linear trend perfectly!")

## 13. Common Pitfalls & Data Leakage

> **Warning:** A major pitfall in time-series feature engineering is looking into the future.

- **Data Leakage in Forecasting:** When creating features for predictive models (e.g., predicting tomorrow's sales), ensure your moving average only uses data up to *today*. If you use pandas `rolling(center=True)`, you will leak future target data into your historical training set, destroying the model's real-world validity.
- **Over-smoothing:** Using too large of a window (like a 365-day average) will completely destroy valuable periodic signals (like weekly or monthly seasonality) that your model critically needs to make accurate short-term predictions.

## 14. Summary and Key Takeaways

- **Purpose:** Smoothing isolates the true signal from random noise, significantly improving model generalization and pattern visibility.
- **SMA (Simple):** Best for broad trend identification, but heavily lags behind sudden, real changes.
- **WMA (Weighted):** Reduces lag by emphasizing recent data while still strictly operating inside a fixed mathematical window.
- **EMA (Exponential):** The most responsive method, decaying older weights exponentially without a strict window cutoff. Excellent for financial data and highly volatile systems.
- **Hyperparameters:** The window size (or span in EMA) is a critical hyperparameter. Small windows maintain volatility; large windows extract macro-trends but lose fine details.

In [ ]:
print("---------------------------------------------------")
print("Notebook Execution Complete.")
print("You have successfully mastered Data Smoothing techniques!")
print("---------------------------------------------------")